# Forward Curve Evaluation

This notebook compares multiple forward-curve recipes (PCHIP and Kalman) using
all available evaluation metrics:

- Put-call parity diagnostics (options vs fitted forwards)
- Snapshot-level WMAE and calendar checks
- Leave-one-expiry-out (LOEO) cross validation

> **Important:** When binning orderbooks you must apply `finalize_binning`
> immediately after `bin` so that `timeMs` matches the bin boundary. The
> `prepare_options` helper already enforces this ordering.

## Imports

In [9]:
from datetime import date
from functools import partial
from pathlib import Path
from typing import Dict

import numpy as np
import polars as pl

from okx.store import OrderbookStore
from okx.recipes.forwards import build_forwards_pchip, build_forwards_kalman, prepare_pillars
from forwards import (
    load_matched_options,
    evaluate_forward_parity,
    wmae_pillar_fit,
    evaluate_curve_snapshot,
    loeo_error,
)
from forwards.parity import compute_option_parity_table, summarize_option_parity
from forwards.pchip import PCHIPCurve, reconstruct_forward, fit_pchip_curve
from forwards.kalman_ns import NSCarryState, reconstruct_ns_forward


## Store Configuration

Update the paths below to point at your local data root and manifest.

In [ ]:
store = OrderbookStore(
    data_root="data/okx",
    manifest_path="data/okx/manifest.sqlite"
)


## Evaluation Parameters

In [11]:
dates = [date(2025, 9, 2)]  # adjust to a range with data
inst_family = "BTC-USD"
binning = "1m"
min_time_to_expiry_hours = 2.0
min_moneyness = 0.95
max_moneyness = 1.05

recipes: Dict[str, callable] = {
    "pchip": build_forwards_pchip,
    "kalman": build_forwards_kalman,
}


## Helper Functions

In [ ]:
def configure_recipe(recipe_fn, unique_times=None):
    if unique_times is not None:
        return partial(recipe_fn, inst_family=inst_family, unique_times=unique_times, verbose=False)
    return partial(recipe_fn, inst_family=inst_family, binning=binning, verbose=False)


def run_parity(recipe_name: str, recipe_fn):
    print(f"=== Put-call parity :: {recipe_name.upper()} ===")
    df_options = load_matched_options(
        store=store,
        dates=dates,
        inst_family=inst_family,
        forwards_recipe=recipe_fn,
        binning=binning,
        min_time_to_expiry_hours=min_time_to_expiry_hours,
        verbose=True,
    )
    
    df_parity = compute_option_parity_table(
        df_options,
        min_moneyness=min_moneyness,
        max_moneyness=max_moneyness,
    )
    summary = summarize_option_parity(df_parity)
    display(summary)
    return df_options, df_parity, summary


def _reconstruct_curve(curve_type: str, curve_df: pl.DataFrame, T: pl.Series) -> tuple:
    if curve_type == 'pchip':
        curve = PCHIPCurve.from_polars(curve_df)
        F_bid, F_ask = reconstruct_forward(curve, T)
        return F_bid, F_ask
    elif curve_type == 'kalman':
        state = NSCarryState.from_polars(curve_df)
        F_bid = reconstruct_ns_forward(state, T, use_bid=True)
        F_ask = reconstruct_ns_forward(state, T, use_bid=False)
        return F_bid, F_ask
    else:
        raise ValueError(f"Unsupported curve type: {curve_type}")


def compute_snapshot_metrics(recipe_name: str, recipe_fn, curve_type: str):
    print(f"=== Pillar diagnostics :: {recipe_name.upper()} ===")
    pillars_df = prepare_pillars(
        store,
        inst_family,
        dates,
        binning,
        min_time_to_expiry_hours=min_time_to_expiry_hours,
    ).collect()
    if pillars_df.is_empty():
        raise RuntimeError("prepare_pillars returned no data")

    forward_recipe = configure_recipe(recipe_fn)
    cache_name = f"eval_{recipe_name}_{binning or 'full'}"
    curves_df = store.get_derived(forward_recipe, dates=dates, cache_name=cache_name, verbose=False).collect()

    metrics_rows = []
    for time_ms in pillars_df['timeMs'].unique().sort().to_list():
        df_snapshot = pillars_df.filter(pl.col('timeMs') == time_ms)
        df_curve = curves_df.filter(pl.col('timeMs') == time_ms)
        if df_curve.is_empty() or len(df_snapshot) < 2:
            continue
        T = df_snapshot['T'].to_numpy()
        F_bid_obs = np.exp(df_snapshot['ln_bid_1_px'].to_numpy())
        F_ask_obs = np.exp(df_snapshot['ln_ask_1_px'].to_numpy())
        F_bid_pred, F_ask_pred = _reconstruct_curve(curve_type, df_curve, T)
        snapshot_metrics = evaluate_curve_snapshot(
            T,
            F_bid_obs,
            F_ask_obs,
            F_bid_pred,
            F_ask_pred,
        ).with_columns([
            pl.lit(time_ms).alias('timeMs'),
            pl.lit(recipe_name).alias('recipe'),
        ])
        metrics_rows.append(snapshot_metrics)

    if not metrics_rows:
        return pl.DataFrame()

    metrics_df = pl.concat(metrics_rows)
    summary = (
        metrics_df
        .group_by(['recipe', 'side'])
        .agg(pl.col('value').mean().alias('avg_wmae_bps'))
    )
    display(summary)
    return metrics_df


def run_loeo(recipe_name: str, recipe_fn):
    print(f"=== LOEO :: {recipe_name.upper()} ===")
    kwargs = {
        'inst_family': inst_family,
        'binning': binning,
        'min_time_to_expiry_hours': min_time_to_expiry_hours,
    }
    loeo_df = loeo_error(store, dates, recipe_fn, kwargs)
    if loeo_df.is_empty():
        print("LOEO result is empty (maybe insufficient pillars)")
        return loeo_df
    summary = loeo_df.filter(pl.col('success')).select([
        pl.mean('error_mid_bps').alias('mean_error_mid_bps'),
        pl.col('error_mid_bps').std().alias('std_error_mid_bps'),
    ])
    display(summary)
    return loeo_df


## Run Full Evaluation

In [14]:
parity_results = {}
snapshot_metrics = {}
loeo_results = {}

for name, fn in recipes.items():
    df_opts, df_parity, parity_summary = run_parity(name, fn)
    parity_results[name] = {
        'options': df_opts,
        'parity': df_parity,
        'summary': parity_summary,
    }

    curve_type = 'pchip' if 'pchip' in fn.__name__.lower() else 'kalman'
    metrics_df = compute_snapshot_metrics(name, fn, curve_type)
    snapshot_metrics[name] = metrics_df

    loeo_df = run_loeo(name, fn)
    loeo_results[name] = loeo_df


=== Put-call parity :: PCHIP ===
Time taken to fetch options: 0:00:00.013037
Time taken to fetch forwards: 0:00:00.002472
Time taken to build forward lookup: 0:00:00.003618


Matching forwards to options: 100%|██████████| 1439/1439 [00:00<00:00, 4188.30it/s]

Time taken to match forwards to options: core 0:00:00.385005, incl. setup 0:00:00.385028 (matched 642,183 / 642,183 options)


n_pairs,error_mid_bps_mean,error_mid_bps_std,error_bid_bps_mean,error_ask_bps_mean,call_spread_bps_mean,put_spread_bps_mean
u32,f64,f64,f64,f64,f64,f64
109782,-5.971488,246.710909,-5.987382,-5.956077,564.601382,470.80967


<sys>:0: UserWarning: Sortedness of columns cannot be checked when 'by' groups provided


...

n_pairs,error_mid_bps_mean,error_mid_bps_std,error_bid_bps_mean,error_ask_bps_mean,call_spread_bps_mean,put_spread_bps_mean
u32,f64,f64,f64,f64,f64,f64
109762,-5.496837,246.550872,-5.726113,-5.268055,566.651922,470.839068


=== Pillar diagnostics :: KALMAN ===


recipe,side,avg_wmae_bps
str,str,f64
"""kalman""","""bid""",2.02994
"""kalman""","""mid""",1.929299
"""kalman""","""ask""",1.864903


=== LOEO :: KALMAN ===


mean_error_mid_bps,std_error_mid_bps
f64,f64
4.593613,4.806678


## Compare WMAE Summaries

In [23]:
wmae_rows = []
for name, metrics_df in snapshot_metrics.items():
    if metrics_df.is_empty():
        continue
    row = metrics_df.group_by('side').agg(pl.col('value').mean().alias('mean_wmae_bps'))
    row = row.with_columns([
        pl.lit(name).alias('recipe'),
    ])
    wmae_rows.append(row)

if wmae_rows:
    wmae_summary = pl.concat(wmae_rows)
    print(wmae_summary)
else:
    print("No WMAE metrics available.")


shape: (6, 3)
┌──────┬───────────────┬────────┐
│ side ┆ mean_wmae_bps ┆ recipe │
│ ---  ┆ ---           ┆ ---    │
│ str  ┆ f64           ┆ str    │
╞══════╪═══════════════╪════════╡
│ ask  ┆ 5.489588      ┆ pchip  │
│ mid  ┆ 5.489512      ┆ pchip  │
│ bid  ┆ 5.495474      ┆ pchip  │
│ bid  ┆ 2.02994       ┆ kalman │
│ ask  ┆ 1.864903      ┆ kalman │
│ mid  ┆ 1.929299      ┆ kalman │
└──────┴───────────────┴────────┘


## LOEO Error Comparison

In [24]:
loeo_rows = []
for name, df in loeo_results.items():
    if df.is_empty():
        continue
    loeo_rows.append(
        df.filter(pl.col('success')).select([
            pl.lit(name).alias('recipe'),
            pl.mean('error_mid_bps').alias('mean_error_mid_bps'),
            pl.col('error_mid_bps').max().alias('max_error_mid_bps'),
        ])
    )

if loeo_rows:
    print(pl.concat(loeo_rows))
else:
    print("LOEO results unavailable.")


shape: (2, 3)
┌────────┬────────────────────┬───────────────────┐
│ recipe ┆ mean_error_mid_bps ┆ max_error_mid_bps │
│ ---    ┆ ---                ┆ ---               │
│ str    ┆ f64                ┆ f64               │
╞════════╪════════════════════╪═══════════════════╡
│ pchip  ┆ 4.489079           ┆ 27.890003         │
│ kalman ┆ 4.593613           ┆ 23.050902         │
└────────┴────────────────────┴───────────────────┘


## Inspect Parity and WMAE Extremes

In [19]:
def show_top_entries(df: pl.DataFrame, column: str, n: int = 5):
    if df.is_empty():
        return pl.DataFrame()
    return df.sort(column, descending=True).head(n)

for name, res in parity_results.items():
    print(f"Top parity errors :: {name.upper()}")
    display(show_top_entries(res['parity'], 'error_mid_bps'))

for name, metrics_df in snapshot_metrics.items():
    print(f"Worst WMAE snapshots :: {name.upper()}")
    if metrics_df.is_empty():
        print("No metrics recorded.")
        continue
    display(show_top_entries(metrics_df, 'value'))


Top parity errors :: PCHIP


timeMs,expiry_dt,strike,T,moneyness,call_bid_1_px,call_ask_1_px,F_bid,F_ask,put_bid_1_px,put_ask_1_px,F_mid,F_implied_bid,F_implied_ask,F_implied_mid,error_bid_bps,error_ask_bps,error_mid_bps,call_spread_bps,put_spread_bps
i64,i64,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
1756781640000,1774598400000,120000,0.564966,1.049992,118.487132,120.681338,114286.081948,114287.0382,171.69663,174.439388,114286.560074,119944.047743,119948.984707,119946.516225,483.205696,483.533622,483.369661,183.486239,158.478605
1756781700000,1774598400000,120000,0.564964,1.049982,118.489372,120.683619,114287.263443,114288.06449,171.699877,174.991248,114287.663966,119943.498124,119948.983743,119946.240933,483.056493,483.443742,483.25012,183.486239,189.873418
1756841880000,1764316800000,118000,0.237028,1.049992,60.80737,61.912959,112379.215199,112384.352285,114.981209,116.086798,112381.783742,117944.720572,117946.93175,117945.826161,483.370428,483.100791,483.235607,180.18018,95.69378
1756837560000,1764316800000,118000,0.237165,1.049993,61.349612,62.455011,112378.517699,112384.944407,115.514135,117.172232,112381.731053,117944.17738,117946.940876,117945.559128,483.38644,483.048878,483.217655,178.571429,142.517815
1756837620000,1764316800000,118000,0.237163,1.049995,61.349583,61.902282,112376.921434,112385.989197,116.066778,117.172176,112381.455316,117944.177407,117945.835503,117945.006455,483.528486,482.862195,483.195333,89.686099,94.78673


Top parity errors :: KALMAN


timeMs,expiry_dt,strike,T,moneyness,call_bid_1_px,call_ask_1_px,F_bid,F_ask,put_bid_1_px,put_ask_1_px,F_mid,F_implied_bid,F_implied_ask,F_implied_mid,error_bid_bps,error_ask_bps,error_mid_bps,call_spread_bps,put_spread_bps
i64,i64,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
1756831560000,1764316800000,118000,0.237355,1.049993,60.797028,61.902428,112381.602951,112381.704617,115.514352,117.172453,112381.653784,117943.624575,117946.388076,117945.006325,483.065032,483.29029,483.177662,180.18018,142.517815
1756776300000,1758268800000,115000,0.047327,1.049997,14.736607,15.282407,109524.081513,109524.181846,68.225031,69.862432,109524.131679,114944.874175,114947.057376,114945.965775,483.082109,483.262881,483.172495,363.636364,237.15415
1756827120000,1761897600000,118000,0.160784,1.049996,42.781643,43.892854,112381.353822,112381.454956,97.786612,100.009035,112381.404389,117942.772608,117946.106242,117944.439425,483.014965,483.28861,483.151789,256.410256,224.719101
1756774200000,1758873600000,115000,0.066572,1.049999,19.623249,20.168339,109523.856422,109523.956886,74.132274,75.222455,109523.906654,114944.400795,114946.036065,114945.21843,483.061477,483.19457,483.128024,273.972603,145.985401
1756808700000,1757664000000,116000,0.027121,1.049997,7.718764,8.270104,110476.453455,110476.553644,61.750108,64.506809,110476.503549,115943.211954,115946.519996,115944.865975,482.981118,483.257361,483.11924,689.655172,436.681223


Worst WMAE snapshots :: PCHIP


metric_name,value,mae_bps,max_error_bps,n_pillars,side,timeMs,recipe
str,f64,f64,f64,i64,str,i64,str
"""wmae_pillar_fit""",51.952234,51.952234,52.878446,8,"""bid""",1756820280000,"""pchip"""
"""wmae_pillar_fit""",51.883328,51.883328,52.878422,8,"""mid""",1756820280000,"""pchip"""
"""wmae_pillar_fit""",51.814446,51.814446,52.878397,8,"""ask""",1756820280000,"""pchip"""
"""wmae_pillar_fit""",47.307225,47.307225,48.101701,8,"""ask""",1756820760000,"""pchip"""
"""wmae_pillar_fit""",47.295959,47.295959,47.964088,8,"""mid""",1756820760000,"""pchip"""


Worst WMAE snapshots :: KALMAN


metric_name,value,mae_bps,max_error_bps,n_pillars,side,timeMs,recipe
str,f64,f64,f64,i64,str,i64,str
"""wmae_pillar_fit""",3.854991,3.854991,15.129273,8,"""bid""",1756774440000,"""kalman"""
"""wmae_pillar_fit""",3.793632,3.793632,14.405758,8,"""bid""",1756818360000,"""kalman"""
"""wmae_pillar_fit""",3.713019,3.713019,9.4909,8,"""bid""",1756820580000,"""kalman"""
"""wmae_pillar_fit""",3.532381,3.532381,11.228992,8,"""bid""",1756782000000,"""kalman"""
"""wmae_pillar_fit""",3.060364,3.060364,8.017392,8,"""bid""",1756848780000,"""kalman"""


---

You can now adjust dates, moneyness bands, or introduce additional
recipes. The helpers above ensure every evaluation metric runs across the
entire selected timeframe.